In [ ]:
import sys
sys.path.insert(0, '..')
from support.paths import resolve
import os
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
from joblib import Parallel, delayed

In [ ]:
%run ../0_Config/0_variables.ipynb

In [ ]:
"""
Fast high-accuracy NEM price forecaster.

Method
------
Spike-aware direct multi-horizon LightGBM ensemble with seven components
per forecast horizon:

  1. BASE L1 model   – regression_l1 on arcsinh(clip(y, p97)). Clipping the
                       target at the 97th percentile removes extreme-spike
                       noise from the base model's loss, giving it a clean
                       training signal for the majority of intervals.

  2. BASE L2 model   – regression (MSE) on arcsinh(y), uncapped target.
                       Blending L1+L2 (Lago et al. 2021) reduces both MAE
                       and RMSE simultaneously. Per-horizon blend weight α
                       tuned on the validation set.

  3. SPIKE classifier – binary LightGBM estimating P(price > SPIKE_THRESHOLD).
                        Trained with scale_pos_weight to handle class imbalance.

  4. SPIKE regressor  – regression_l1 on arcsinh(y) (full range, no clipping).
                        Spike rows receive 10x sample weight.

  5. SPIKE quantile   – Predicts the 90th percentile for conservative spike ceilings. 
                        P90 quantile regression on arcsinh(y) for a
                        conservative spike ceiling estimate.

  6. DIP classifier   – binary LightGBM estimating P(price < SPIKE_THRESHOLD).
                        Trained with scale_pos_weight to handle class imbalance.

  7. DIP regressor    – regression_l1 on arcsinh(y) (full range, no clipping).
                        DIP rows receive 7x sample weight.

"""


In [ ]:
# Load hyperparameters
with open("Data/hyperparameters.json", "r", encoding="utf-8") as f:
    MODEL_HYPERPARAMETERS = json.load(f)

full_range_regressor_clipped_MAE_loss_params = MODEL_HYPERPARAMETERS["full_range_regressor_clipped_MAE_loss"]
full_range_regressor_unclipped_RMSE_loss_params = MODEL_HYPERPARAMETERS["full_range_regressor_unclipped_RMSE_loss"]
positive_spike_classifier_binary_loss_params = MODEL_HYPERPARAMETERS["positive_spike_classifier_binary_loss"]
positive_spike_regressor_unclipped_mae_loss_params = MODEL_HYPERPARAMETERS["positive_spike_regressor_unclipped_mae_loss"]
positive_spike_regressor_unclipped_quantile_loss_params = MODEL_HYPERPARAMETERS["positive_spike_regressor_unclipped_quantile_loss"]
negative_spike_classifer_unclipped_binary_loss_params = MODEL_HYPERPARAMETERS["negative_spike_classifer_unclipped_binary_loss"]
negative_spike_regressor_unclipped_mae_loss_params = MODEL_HYPERPARAMETERS["negative_spike_regressor_unclipped_mae_loss"]

In [ ]:
# LGBM model setup
def LBGM_model_setup(hyperparameters: dict, X_train: pd.DataFrame, y_train: pd.DataFrame, X_validate: pd.DataFrame, y_validate: pd.DataFrame, loss_weights: pd.DataFrame, early_stopping_rounds: int):
    
    model = lgb.LGBMRegressor(**hyperparameters)
    model.fit(
        X_train, 
        y_train,
        sample_weight=loss_weights,
        eval_set=[(X_validate, y_validate)],
        callbacks=[lgb.early_stopping(early_stopping_rounds, verbose=False)],
    )

    return model

In [ ]:
def _train_models_per_horizon(horizon: int):

    MIN_POSITIVE_SPIKE_TRAIN = 20
    MIN_NEGATIVE_SPIKE_TRAIN = 20
    NORMAL_EARLY_STOPPING_ROUNDS = 75
    SPIKE_EARLY_STOPPING_ROUNDS = 50

    target_col = f"h{horizon}"

    # injest derived targets
    derived_targets = _create_y_derivatives_per_target_col(target_col)

    y_train_individual_clipped_transformed = derived_targets['y_train_individual_clipped_transformed']
    y_validation_individual_clipped_transformed = derived_targets['y_validation_individual_clipped_transformed']

    y_train_individual_unclipped_transformed = derived_targets['y_train_individual_unclipped_transformed']
    y_validation_individual_unclipped_transformed = derived_targets['y_validation_individual_unclipped_transformed']

    positive_spike_labels_train = derived_targets['positive_spike_labels_train']
    positive_spike_labels_validate = derived_targets['positive_spike_labels_validate']

    negative_spike_labels_train = derived_targets['negative_spike_labels_train']
    negative_spike_labels_validate = derived_targets['negative_spike_labels_validate']

    y_train_loss_weights = derived_targets['y_train_loss_weights']
    y_train_full_range_loss_weights = derived_targets['y_train_full_range_loss_weights']
    y_train_positive_spike_loss_weights = derived_targets['y_train_positive_spike_loss_weights']
    y_train_negative_spike_loss_weights = derived_targets['y_train_negative_spike_loss_weights']



    # Train each model
    
    full_range_regressor_clipped_MAE_loss = LBGM_model_setup(
        full_range_regressor_clipped_MAE_loss_params,
        X_train,
        y_train_individual_clipped_transformed,
        X_validate,
        y_validation_individual_clipped_transformed,
        y_train_full_range_loss_weights,
        NORMAL_EARLY_STOPPING_ROUNDS
        )
    
    full_range_regressor_unclipped_RMSE_loss = LBGM_model_setup(
        full_range_regressor_unclipped_RMSE_loss_params,
        X_train,
        y_train_individual_unclipped_transformed,
        X_validate,
        y_validation_individual_unclipped_transformed,
        y_train_full_range_loss_weights,
        NORMAL_EARLY_STOPPING_ROUNDS
        )

    positive_spike_classifier_binary_loss = LBGM_model_setup(
        positive_spike_classifier_binary_loss_params,
        X_train,
        positive_spike_labels_train,
        X_validate,
        positive_spike_labels_validate,
        y_train_loss_weights,
        SPIKE_EARLY_STOPPING_ROUNDS
        )
    
    positive_spike_regressor_unclipped_mae_loss = LBGM_model_setup(
        positive_spike_regressor_unclipped_mae_loss_params,
        X_train,
        y_train_individual_unclipped_transformed,
        X_validate,
        y_validation_individual_unclipped_transformed,
        y_train_full_range_loss_weights,
        SPIKE_EARLY_STOPPING_ROUNDS
        )
    
    positive_spike_regressor_unclipped_quantile_loss = LBGM_model_setup(
        positive_spike_regressor_unclipped_quantile_loss_params,
        X_train,
        y_train_individual_unclipped_transformed,
        X_validate,
        y_validation_individual_unclipped_transformed,
        y_train_positive_spike_loss_weights,
        SPIKE_EARLY_STOPPING_ROUNDS
        )

    negative_spike_classifer_unclipped_binary_loss = LBGM_model_setup(
        negative_spike_classifer_unclipped_binary_loss_params,
        X_train,
        negative_spike_labels_train,
        X_validate,
        negative_spike_labels_validate,
        y_train_loss_weights,
        SPIKE_EARLY_STOPPING_ROUNDS
        )
    
    negative_spike_regressor_unclipped_mae_loss = LBGM_model_setup(
        negative_spike_regressor_unclipped_mae_loss_params,
        X_train,
        y_train_individual_unclipped_transformed,
        X_validate,
        y_validation_individual_unclipped_transformed,
        y_train_negative_spike_loss_weights,
        SPIKE_EARLY_STOPPING_ROUNDS
        )

    return {
        'full_range_regressor_clipped_MAE_loss' : full_range_regressor_clipped_MAE_loss,
        'full_range_regressor_unclipped_RMSE_loss' : full_range_regressor_unclipped_RMSE_loss,
        'positive_spike_classifier_binary_loss' : positive_spike_classifier_binary_loss,
        'positive_spike_regressor_unclipped_mae_loss' : positive_spike_regressor_unclipped_mae_loss,
        'positive_spike_regressor_unclipped_quantile_loss' : positive_spike_regressor_unclipped_quantile_loss,
        'negative_spike_classifer_unclipped_binary_loss' : negative_spike_classifer_unclipped_binary_loss,
        'negative_spike_regressor_unclipped_mae_loss' : negative_spike_regressor_unclipped_mae_loss
    }
        
horizon_list = list(range(1, int(os.environ["N_HORIZONS"]) + 1))

trained_models = Parallel(n_jobs=-1, prefer="threads")(
    delayed(_train_models_per_horizon)(h) for h in horizon_list
)

 

In [ ]:
%reset -f